# Agonal Breathing Model — Charts, Diagrams & Explanations

This notebook reproduces the evaluation and training visualizations from the TASH
agonal gasp detector (`audio/models/agonal_detector.joblib`).

**Run from the repo root** (or set `REPO` below):
```bash
cd tash-P7-group2
jupyter notebook notebooks/agonal_model_visualization.ipynb
```

**Requirements:** `matplotlib`, `numpy`, `joblib`, `librosa`, `scikit-learn`, `soundfile`
(already in the project venv).

## 1. Architecture — what the model actually is

The detector is **two-stage** (inspired by UW 2019 / Donahue et al.):

| Stage | Trained? | Role |
|-------|----------|------|
| **Stage A — Gasp classifier** | Yes | Score each **2.5 s** window: *P(contains agonal gasp)* |
| **Stage B — Rate filter** | No (rules) | Over a rolling **30 s** buffer, count gasp hits; classify rate/regularity |

**Stage A algorithm (deployed):**
- `StandardScaler` → `HistGradientBoostingClassifier` (gradient-boosted histogram trees)
- 181-dim features: log-mel (32), MFCC+delta (20), RMS stats, spectral contrast, YIN F0, HPSS percussiveness
- Decision threshold **0.55** (`GASP_THRESH`) to count a gasp event

**Stage B rules (inference only):**
- 3–6 gasps/min + irregular spacing → `AGONAL_SUSPECT`
- No gasps in silent audio → `APNEA` candidate
- Regular 10–25 BPM → `NORMAL`

The diagram in the next code cell draws this pipeline.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np

# Point to repo root (parent of notebooks/)
REPO = Path.cwd()
if (REPO / "audio" / "models").exists():
    pass
elif (REPO.parent / "audio" / "models").exists():
    REPO = REPO.parent
else:
    raise FileNotFoundError("Run from tash-P7-group2 or notebooks/ inside it")

sys.path.insert(0, str(REPO))

MODEL_PATH = REPO / "audio" / "models" / "agonal_detector.joblib"
DEMO_WAV = REPO / "audio" / "test_audio" / "demo_agonal_real.wav"
GASP_THRESH = 0.55

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

bundle = joblib.load(MODEL_PATH)
pipe = bundle["pipeline"]
clf = pipe.named_steps["clf"]
print(f"Loaded: {type(clf).__name__}")
print(f"Features: {bundle['feature_dim']}  Window: {bundle['window_s']}s  SR: {bundle['sample_rate']}")

In [ ]:
# ── Cell 2: Architecture diagram ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")

def box(x, y, w, h, text, color="#e8f4fc", edge="#2b6cb0"):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor=edge, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=9, wrap=True)

def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#444", lw=1.5))

box(0.2, 1.0, 1.6, 1.0, "Mic / WAV\n16 kHz PCM", "#f0f0f0", "#666")
arrow(1.8, 1.5, 2.3, 1.5)
box(2.3, 0.6, 2.0, 1.8, "Stage A (ML)\n2.5 s window\n181 features\nHGBC → P(gasp)", "#fff3cd", "#b8860b")
arrow(4.3, 1.5, 4.8, 1.5)
box(4.8, 1.0, 1.8, 1.0, "Threshold\nP ≥ 0.55", "#f8d7da", "#c0392b")
arrow(6.6, 1.5, 7.1, 1.5)
box(7.1, 0.6, 2.2, 1.8, "Stage B (rules)\n30 s event buffer\nRate + irregularity", "#d4edda", "#27ae60")
arrow(9.3, 1.5, 9.8, 1.5)
box(9.8, 0.8, 1.8, 1.4, "BreathingState\nAGONAL_SUSPECT\nAPNEA / NORMAL", "#e8f4fc", "#2b6cb0")

ax.set_title("Two-stage agonal breathing detector (training vs inference)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Training charts

### Chart A — Train vs held-out metrics (grouped bar chart)

**What it shows:** After training, the script scores the model on:
- **Train split** (23,543 windows) — data the trees were fit on
- **Held-out test** (5,969 windows) — entire patient/source-file groups *excluded* from training

**Why it matters:** A large gap between bars means overfitting. Here the gap is tiny
(accuracy −0.3%, F1 −0.6%), so the model generalizes.

**Split method:** `GroupShuffleSplit` 80/20 — no ICBHI patient or recording leaks across splits.

In [ ]:
# ── Cell 3: Train vs held-out metrics ──────────────────────────────────────
hm = bundle["held_out_metrics"]
train_m = hm["train_metrics"]

metrics = ["Accuracy", "F1", "Precision", "Recall"]
train_vals = [train_m["accuracy"]*100, train_m["f1"]*100, train_m["precision"]*100, train_m["recall"]*100]
test_vals  = [hm["accuracy"]*100, hm["f1"]*100, hm["precision"]*100, hm["recall"]*100]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, train_vals, w, label=f"Train ({train_m['n_train']:,} windows)", color="#5b9bd5")
ax.bar(x + w/2, test_vals,  w, label=f"Held-out test ({hm['n_test']:,} windows)", color="#70ad47")
ax.set_ylabel("Score (%)")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(98, 100.5)
ax.legend()
ax.set_title("Train vs group-held-out test metrics")
for i, (a, b) in enumerate(zip(train_vals, test_vals)):
    ax.text(i - w/2, a + 0.05, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, b + 0.05, f"{b:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()
print(f"Held-out AUC: {hm['auc']:.4f}")

### Chart B — Training loss curve (boosting iterations)

**What it shows:** `HistGradientBoostingClassifier` minimizes **log loss** each boosting
iteration. The curve is stored in `clf.train_score_` after fitting.

**How to read it:**
- Steep drop early = model learning gasp vs non-gasp structure quickly
- Flattening = diminishing returns; **early stopping** halted at iteration **231** (of max 300)
- sklearn stores log loss as negative values (closer to 0 = better)

**Hyperparameters driving this curve:** `learning_rate=0.05`, `max_depth=6`, `max_leaf_nodes=31`

In [ ]:
# ── Cell 4: Learning curve ─────────────────────────────────────────────────
train_loss = np.array(clf.train_score_)
iters = np.arange(1, len(train_loss) + 1)
n_iter = int(getattr(clf, "n_iter_", len(train_loss)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(iters, np.abs(train_loss), color="#2b6cb0", linewidth=2)
ax.axvline(n_iter, color="#c0392b", linestyle="--", label=f"Early stop @ iter {n_iter}")
ax.set_xlabel("Boosting iteration")
ax.set_ylabel("|log_loss| (lower = better fit)")
ax.set_title("HistGradientBoosting training loss curve")
ax.legend()
plt.tight_layout()
plt.show()

print("Classifier hyperparameters:")
for k in ["max_iter", "learning_rate", "max_depth", "max_leaf_nodes", "min_samples_leaf",
          "class_weight", "early_stopping", "validation_fraction"]:
    print(f"  {k}: {clf.get_params()[k]}")

## 3. Evaluation charts (model behavior on audio)

### Chart C — P(gasp) timeline over demo clip

**What it shows:** For `demo_agonal_real.wav`, we slide a **2.5 s window** forward every **0.5 s**
and plot the model's calibrated **P(gasp)** at each position.

**How to read it:**
- **Red dashed line** = threshold 0.55 — above this, Stage A counts a gasp event
- High plateau (0.9–1.0) in seconds 2–14 = gasp-rich region of the clip
- Drop below threshold at 15–20 s = near-silence tail (explains why "last 5 s" doesn't trigger agonal)
- Stage B needs *multiple* threshold crossings over 30 s to emit `AGONAL_SUSPECT`

In [ ]:
# ── Cell 5: Demo WAV P(gasp) timeline ─────────────────────────────────────
import soundfile as sf
from audio.train_agonal_detector import extract_features, WINDOW_N, SR
from audio.stage3_ml_breathing import HOP_A_N

audio, fs = sf.read(DEMO_WAV, dtype="float32")
assert fs == SR

times, probs = [], []
for start in range(0, len(audio) - WINDOW_N + 1, HOP_A_N):
    win = audio[start : start + WINDOW_N]
    p = float(pipe.predict_proba(extract_features(win).reshape(1, -1))[0, 1])
    times.append(start / fs)
    probs.append(p)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, probs, "o-", color="#2b6cb0", markersize=4, label="P(gasp)")
ax.axhline(GASP_THRESH, color="#c0392b", linestyle="--", linewidth=2, label=f"Threshold {GASP_THRESH}")
ax.fill_between(times, GASP_THRESH, 1, where=[p >= GASP_THRESH for p in probs],
                alpha=0.15, color="#c0392b", label="Gasp detected")
ax.set_xlabel("Window start time (seconds)")
ax.set_ylabel("P(gasp)")
ax.set_ylim(0, 1.05)
ax.set_title("Stage A scores on demo_agonal_real.wav (2.5 s window, 0.5 s hop)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

### Chart D — Recall vs SNR (noise robustness)

**What it shows:** Fresh synthetic gasps mixed with car/HVAC noise at different
**signal-to-noise ratios** (SNR in dB). Measures what fraction exceed threshold 0.55.

**How to read it:**
- Left side (−5 dB) = very noisy cabin; recall still ~98%
- Right side (+10 dB and up) = clean gasps; recall 100%
- **Mean P(gasp)** bars show average confidence, not just pass/fail

**Note:** Uses seed 99 (different from training seed 42) — true out-of-distribution test.

⏱ This cell takes ~1–3 minutes (runs `tests/test_generalization.py` logic inline).

In [ ]:
# ── Cell 6: SNR recall curve ───────────────────────────────────────────────
from collections import defaultdict
from audio.train_agonal_detector import (
    extract_features, gen_gasp, gen_car_window, gen_hvac_window,
    _mix_at_snr, SNR_LEVELS_DB,
)

RNG = np.random.default_rng(99)
N_PER = 40  # increase to 60 to match full test
noise_fns = [gen_car_window, gen_hvac_window]

snr_labels, recalls, mean_ps = [], [], []
for snr_db in SNR_LEVELS_DB:
    hits, probs = 0, []
    for _ in range(N_PER):
        g = gen_gasp(rng=RNG)
        noise = noise_fns[int(RNG.integers(len(noise_fns)))](RNG)
        noisy = _mix_at_snr(g, noise, snr_db)
        p = float(pipe.predict_proba(extract_features(noisy).reshape(1, -1))[0, 1])
        probs.append(p)
        hits += int(p >= GASP_THRESH)
    snr_labels.append(f"{snr_db:+.0f} dB")
    recalls.append(100 * hits / N_PER)
    mean_ps.append(100 * np.mean(probs))

fig, ax1 = plt.subplots(figsize=(9, 4))
x = np.arange(len(snr_labels))
ax1.bar(x - 0.2, recalls, 0.4, label="Recall (%)", color="#70ad47")
ax1.bar(x + 0.2, mean_ps, 0.4, label="Mean P(gasp) (%)", color="#5b9bd5")
ax1.set_xticks(x)
ax1.set_xticklabels(snr_labels)
ax1.set_ylabel("Percent")
ax1.set_ylim(0, 105)
ax1.set_xlabel("SNR (signal-to-noise ratio)")
ax1.set_title(f"Gasp detection vs noise (n={N_PER} fresh gasps per SNR, seed 99)")
ax1.legend()
plt.tight_layout()
plt.show()

### Chart E — Score distribution (gasp vs snoring histogram)

**What it shows:** Histogram of P(gasp) for 120 **fresh agonal gasps** vs 120 **snore windows**.

**How to read it:**
- Gasps pile up in bin 0.9–1.0 (far right) — model is very confident
- Snoring piles up in bin 0.0–0.1 (far left) — correctly rejected
- Clean separation at 0.55 threshold = low false-positive risk on snoring

This is the most important "is the classifier calibrated?" chart.

In [ ]:
# ── Cell 7: Score distribution histogram ───────────────────────────────────
from audio.train_agonal_detector import gen_snore_window

RNG = np.random.default_rng(99)
gasp_probs, snore_probs = [], []
for _ in range(120):
    gasp_probs.append(float(pipe.predict_proba(
        extract_features(gen_gasp(rng=RNG)).reshape(1, -1))[0, 1]))
    snore_probs.append(float(pipe.predict_proba(
        extract_features(gen_snore_window(rng=RNG)).reshape(1, -1))[0, 1]))

bins = np.linspace(0, 1, 11)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(gasp_probs, bins=bins, alpha=0.7, label="Agonal gasps (positive)", color="#c0392b")
ax.hist(snore_probs, bins=bins, alpha=0.7, label="Snoring (negative)", color="#7f8c8d")
ax.axvline(GASP_THRESH, color="black", linestyle="--", linewidth=2, label=f"Threshold {GASP_THRESH}")
ax.set_xlabel("P(gasp)")
ax.set_ylabel("Window count")
ax.set_title("Score distribution: gasps vs snoring (120 each, seed 99)")
ax.legend()
plt.tight_layout()
plt.show()

### Chart F — False-positive rate on hard negatives

**What it shows:** For 8 **negative** audio classes (normal breath, snoring, car, speech, etc.),
what fraction of windows falsely score above 0.55?

**How to read it:**
- **FP rate** should be 0% for all — any bar here means the model cried wolf
- **Mean P(gasp)** shows how close negatives get to the threshold (should stay near 0)

In a car safety system, FP rate on snoring/speech is critical — one false alarm erodes trust.

In [ ]:
# ── Cell 8: Hard negatives FP rate ─────────────────────────────────────────
from audio.train_agonal_detector import (
    gen_normal_breath_window, gen_slow_breath_window, gen_speech_window,
    gen_radio_window, gen_hvac_window, gen_silence_window,
)

NEG_CLASSES = {
    "Normal breath": gen_normal_breath_window,
    "Slow breath": gen_slow_breath_window,
    "Snoring": gen_snore_window,
    "Car noise": gen_car_window,
    "Speech": gen_speech_window,
    "HVAC": gen_hvac_window,
    "Radio": gen_radio_window,
    "Silence": gen_silence_window,
}

RNG = np.random.default_rng(99)
N = 40
labels, fp_rates, mean_ps = [], [], []
for name, fn in NEG_CLASSES.items():
    fps, probs = 0, []
    for _ in range(N):
        w = fn(rng=RNG)
        p = float(pipe.predict_proba(extract_features(w).reshape(1, -1))[0, 1])
        probs.append(p)
        fps += int(p >= GASP_THRESH)
    labels.append(name)
    fp_rates.append(100 * fps / N)
    mean_ps.append(100 * np.mean(probs))

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(labels))
ax.barh(y - 0.2, fp_rates, 0.35, label="FP rate (%)", color="#e74c3c")
ax.barh(y + 0.2, mean_ps, 0.35, label="Mean P(gasp) (%)", color="#3498db")
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("Percent")
ax.set_title(f"Hard negatives — false positive check (n={N} per class)")
ax.legend()
plt.tight_layout()
plt.show()

### Chart G — Stress / borderline cases

**What it shows:** Edge cases beyond standard training:
- **Faint gasps** (5% amplitude) — hardest positive
- **−10 dB SNR** — noisier than training range
- **Gasp + snoring overlay** — realistic confusion case
- **Loud snoring / speech** — must stay negative

**How to read it:** Recall on faint gasps (~92%) is the weak spot; everything else is near perfect on synthetic OOD data.

In [ ]:
# ── Cell 9: Stress cases ───────────────────────────────────────────────────
from audio.train_agonal_detector import _mix_at_snr

RNG = np.random.default_rng(99)
N = 60

def eval_cases(name, windows, true_positive=True):
    hits, probs = 0, []
    for w in windows:
        p = float(pipe.predict_proba(extract_features(w).reshape(1, -1))[0, 1])
        probs.append(p)
        pred_pos = p >= GASP_THRESH
        if true_positive:
            hits += int(pred_pos)
        else:
            hits += int(not pred_pos)  # TN for negatives
    recall = 100 * hits / len(windows)
    return recall, float(np.mean(probs))

cases = []
# Faint gasps
w = [gen_gasp(rng=RNG) * 0.05 for _ in range(N)]
cases.append(("Faint gasp (5% amp)", *eval_cases("faint", w, True)))
# -10 dB SNR
w = [_mix_at_snr(gen_gasp(rng=RNG), gen_car_window(RNG), -10.0) for _ in range(N)]
cases.append(("Gasp at -10 dB SNR", *eval_cases("-10db", w, True)))
# Gasp + snore
w = [np.clip(gen_gasp(rng=RNG) + gen_snore_window(RNG), -1, 1).astype(np.float32) for _ in range(N)]
cases.append(("Gasp + snoring", *eval_cases("mix", w, True)))
# Loud snore (neg)
w = [gen_snore_window(rng=RNG) * 2.0 for _ in range(N)]
cases.append(("Loud snoring (neg)", *eval_cases("loud_snore", w, False)))
# Speech (neg)
w = [gen_speech_window(rng=RNG) for _ in range(N)]
cases.append(("Speech (neg)", *eval_cases("speech", w, False)))

names = [c[0] for c in cases]
recalls = [c[1] for c in cases]
mean_ps = [c[2] for c in cases]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(names))
ax.bar(x - 0.2, recalls, 0.4, label="Recall / TN rate (%)", color="#27ae60")
ax.bar(x + 0.2, [p*100 for p in mean_ps], 0.4, label="Mean P(gasp) (%)", color="#2980b9")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha="right")
ax.set_ylabel("Percent")
ax.set_ylim(0, 105)
ax.set_title("Stress / borderline cases (n=60 each)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Summary table — all charts at a glance

| Chart | Type | What it answers |
|-------|------|-----------------|
| Architecture diagram | Flow | How Stage A (ML) + Stage B (rules) connect |
| Train vs test bars | Bar | Is the model overfitting? |
| Learning curve | Line | Did boosting converge? When did early stop fire? |
| P(gasp) timeline | Line | Where in the demo clip does the model see gasps? |
| SNR recall | Bar | Does cabin noise break detection? |
| Score histogram | Histogram | Are positives and negatives well separated? |
| Hard negatives | Horizontal bar | Does snoring/speech trigger false alarms? |
| Stress cases | Bar | Worst-case OOD performance |